![](https://github.com/mccode-dev/PaNRAID/raw/main/images/toplogo-diadem.png)

# From a McStas instrument to a simulation dataset ready for AI

Let's think together on what is needed for the generation of a large dataset of simulation for AI purposes using **McStas**. We will need to have several aspects into consideration, amongst them: 

- **purpose**: from an AI point of view, supervised or unsupervised? Any objective in particular? 

- **structure**: data, metadata specification, labels, targets, etc.

- **parameter space**: what to vary and how. 

- **Format**: NeXus, HDF5, csv, etc.  May be related to posterior usage.

Many of the considerations that we will work on and the example we are using is based on [the following paper: "Learning from virtual experiments to assist users of Small Angle Neutron Scattering in model selection" Robledot et al. 2024](https://www.nature.com/articles/s41598-024-65712-y)

In [ ]:
from pathlib import Path
import itertools
import json
import math
import shlex
import shutil
import subprocess

import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import qmc

ROOT = Path.cwd()
print(f'Working directory: {ROOT}')
print(f'mcrun available: {shutil.which("mcrun") is not None}')

## Define the scientific learning problem first

A simulation parameter can play one of several roles. Keeping these roles explicit prevents target leakage.

| Role | KWS example | Typical ML treatment |
|---|---|---|
| Instrument configuration | wavelength, slit setting, detector distance | input metadata or nuisance variation |
| Sample model | sphere, core–shell, linear pearls | class label |
| Sample parameter | radius, shell thickness, polydispersity | regression target or nuisance variation |
| Monte Carlo setting | neutron histories, random seed | fidelity/noise control; usually not a target |

Write the intended model input and output before launching a large sweep. For example: *infer sample family and particle radius from a 2D detector image, robustly across wavelength and collimation settings*.

In [ ]:
sample_instruments = [
    ROOT / 'kws_sphere.instr',
    ROOT / 'kws_core_shell.instr',
    ROOT / 'kws_linear_pearls.instr',
]
print('Sample instrument definitions used in this lesson:')
for path in sample_instruments:
    print(f' - {path.name}: {"found" if path.exists() else "missing"}')

# The DEFINE INSTRUMENT declaration contains the parameters that mcrun may vary.
print('\nBeginning of each instrument definition:')
for path in sample_instruments:
    declaration = '\n'.join(path.read_text().splitlines()[:12])
    print(f'\n--- {path.name} ---\n{declaration}')

### Constructing sweep commands from the instruments

Begin with the `DEFINE INSTRUMENT` declaration in each `.instr` file. It tells us which values can be supplied at run time. For example, the sphere instrument exposes `radius` and `pd_radius`; the core–shell instrument additionally exposes `thickness`; and the linear-pearls instrument exposes `edge_sep`.

A sweep command is assembled from four decisions:

- **Instrument:** the `.instr` file to simulate.
- **Fidelity:** `-n` gives the neutron histories in every simulation.
- **Execution and output:** `--mpi` selects MPI ranks and `--format=NeXus` requests HDF5/NeXus output.
- **Parameter design:** `-M` requests the Cartesian product, `-N` gives the number of values per range, and `parameter=minimum,maximum` defines each range.

For `d` scanned parameters and `N` values per parameter, multi-run mode produces $N^d$ simulations. This exponential growth must be calculated before starting the command. The parameter bounds below are teaching choices; they should be justified scientifically for a real dataset.

> Check `mcrun --help` for the installed McStas version. Scheduler and MPI options vary across systems.

In [ ]:
sweep_specs = {
    'sphere': {
        'instrument': 'kws_sphere.instr',
        'levels': 20,
        'ranges': {'radius': (10, 100), 'pd_radius': (0.0, 0.1)},
    },
    'core_shell': {
        'instrument': 'kws_core_shell.instr',
        'levels': 5,
        'ranges': {'thickness': (10, 100), 'radius': (10, 100), 'pd_radius': (0.0, 0.1)},
    },
    'linear_pearls': {
        'instrument': 'kws_linear_pearls.instr',
        'levels': 5,
        'ranges': {'radius': (10, 100), 'edge_sep': (100, 400), 'pd_radius': (0.0, 0.1)},
    },
}

histories = 10_000_000
for name, spec in sweep_specs.items():
    dimensions = len(spec['ranges'])
    count = spec['levels'] ** dimensions
    print(f'{name:14s}: {count:4d} simulations = {spec["levels"]}^{dimensions}')

total = sum(spec['levels'] ** len(spec['ranges']) for spec in sweep_specs.values())
print(f'Nominal total: {total:,} simulations and {total * histories:,} neutron histories')

In [ ]:
rng = np.random.default_rng(42)
samples_per_class = 500

design = []

for index in range(samples_per_class):
    design.append({
        "simulation_id": f"sphere_{index:06d}",
        "sample_family": "sphere",
        "instrument": "kws_sphere.instr",
        "radius": rng.uniform(10, 100),
        "pd_radius": rng.uniform(0.0, 0.1),
    })

    design.append({
        "simulation_id": f"core_shell_{index:06d}",
        "sample_family": "core_shell",
        "instrument": "kws_core_shell.instr",
        "radius": rng.uniform(10, 100),
        "thickness": rng.uniform(10, 100),
        "pd_radius": rng.uniform(0.0, 0.1),
    })

    design.append({
        "simulation_id": f"linear_pearls_{index:06d}",
        "sample_family": "linear_pearls",
        "instrument": "kws_linear_pearls.instr",
        "radius": rng.uniform(10, 100),
        "edge_sep": rng.uniform(100, 400),
        "pd_radius": rng.uniform(0.0, 0.1),
    })

## Design the parameter space

A Cartesian grid is transparent but scales exponentially and introduces strong correlations between neighboring points. Random, Latin-hypercube, or low-discrepancy designs cover higher-dimensional spaces more efficiently. Distributions should reflect scientific use: log-uniform sampling is often preferable for positive parameters spanning orders of magnitude.

Always include physically valid bounds and constraints (for example, shell thickness must be positive). Reserve boundary and out-of-distribution regions deliberately for evaluation.

In [ ]:
# A small, explicit grid useful for debugging before an expensive production run
debug_space = {
    'radius': [10.0, 55.0, 100.0],
    'pd_radius': [0.0, 0.05, 0.1],
}
debug_design = []

for radius in debug_space["radius"]:
    for pd_radius in debug_space["pd_radius"]:
        debug_design.append({
            "radius": radius,
            "pd_radius": pd_radius,
        })
        
print(f'{len(debug_design)} design points')
debug_design[:4]

In [ ]:
# A randomized design. Save it: the table, not only a seed, is the definitive record.
rng = np.random.default_rng(20260912)
n_design = 100
random_design = {
    'radius': np.exp(rng.uniform(np.log(10.0), np.log(100.0), n_design)),
    'pd_radius': rng.uniform(0.0, 0.1, n_design),
    'Lam': rng.uniform(4.0, 10.0, n_design),  # example instrument variation
}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].scatter(random_design['radius'], random_design['pd_radius'], s=12, alpha=.7)
axes[0].set(xlabel='radius', ylabel='pd_radius', xscale='log')
axes[1].hist(random_design['Lam'], bins=15)
axes[1].set(xlabel='wavelength (Å)', ylabel='count')
plt.tight_layout()

### Latin-hypercube sampling alternative

Independent random sampling can leave gaps and clusters. Latin-hypercube sampling divides every parameter's probability range into equally probable intervals and uses each interval once. This improves one-dimensional coverage without constructing an exponentially large Cartesian grid.

Latin-hypercube sampling is provided by SciPy's `scipy.stats.qmc` module. Scikit-learn provides random parameter sampling for model search, but it does not provide a general Latin-hypercube design generator. `qmc.LatinHypercube` creates points in the unit cube and `qmc.scale` maps them to physical bounds.

In [ ]:
parameter_names = ['radius', 'pd_radius', 'Lam']
lower_bounds = [10.0, 0.0, 4.0]
upper_bounds = [100.0, 0.1, 10.0]

lhs_sampler = qmc.LatinHypercube(d=len(parameter_names), seed=20260912)
lhs_unit = lhs_sampler.random(n=n_design)
lhs_scaled = qmc.scale(lhs_unit, lower_bounds, upper_bounds)
lhs_design = [dict(zip(parameter_names, row)) for row in lhs_scaled]

print(f'Generated {len(lhs_design)} Latin-hypercube points')
lhs_design[:3]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True, sharey=True)
axes[0].scatter(random_design['radius'], random_design['pd_radius'], s=18, alpha=.75)
axes[0].set_title('Independent random sampling')
axes[1].scatter([row['radius'] for row in lhs_design],
                [row['pd_radius'] for row in lhs_design], s=18, alpha=.75)
axes[1].set_title('Latin-hypercube sampling')
for ax in axes:
    ax.set(xlabel='radius', ylabel='pd_radius', xlim=(10, 100), ylim=(0, 0.1))
    ax.grid(alpha=.2)
plt.tight_layout()

random_unit = np.column_stack([
    (random_design['radius'] - 10.0) / 90.0,
    random_design['pd_radius'] / 0.1,
    (random_design['Lam'] - 4.0) / 6.0,
])
print('Centered discrepancy; lower indicates more even unit-cube coverage:')
print(f'  independent random: {qmc.discrepancy(random_unit):.5f}')
print(f'  Latin hypercube:    {qmc.discrepancy(lhs_unit):.5f}')

## Generate the sweep commands

The function below converts a sweep specification into an argument list suitable for `subprocess.run`. Argument lists are safer than manually concatenated shell commands because quoting is unambiguous. It also checks that the instrument exists, the level count is positive, every range has increasing bounds, and every varied parameter name occurs in the instrument source.

The textual command printed with `shlex.join` is the same command that will later be placed in a shell script.

In [ ]:
def make_multirun_command(spec, histories=10_000_000, mpi=6):
    instrument = ROOT / spec['instrument']
    levels = int(spec['levels'])
    ranges = spec['ranges']
    if not instrument.is_file():
        raise FileNotFoundError(instrument)
    if levels < 1 or histories < 1 or mpi < 1:
        raise ValueError('levels, histories and mpi must be positive')
    source = instrument.read_text()
    for parameter, (low, high) in ranges.items():
        if parameter not in source:
            raise ValueError(f'{parameter!r} was not found in {instrument.name}')
        if low >= high:
            raise ValueError(f'{parameter}: lower bound must be smaller than upper bound')
    command = [
        'mcrun', instrument.name, f'-n{int(histories)}', f'--mpi={int(mpi)}',
        '--format=NeXus', '-M', f'-N{levels}',
    ]
    command.extend(f'{name}={low},{high}' for name, (low, high) in ranges.items())
    return command

sweep_commands = {
    name: make_multirun_command(spec, histories=histories, mpi=6)
    for name, spec in sweep_specs.items()
}
for name, command in sweep_commands.items():
    print(f'{name}:\n  {shlex.join(command)}\n')

In [ ]:
def shell_script(command):
    return (
        '#!/usr/bin/env bash\n'
        'set -euo pipefail\n\n'
        '# Generated from sweep_specs in this notebook.\n'
        f'{shlex.join(command)}\n'
    )

generated_scripts = {
    f'run_generated_{name}.sh': shell_script(command)
    for name, command in sweep_commands.items()
}
for filename, content in generated_scripts.items():
    print(f'--- {filename} ---\n{content}')

### Write one `.sh` file per sample instrument

Each generated script starts with a portable Bash interpreter declaration and `set -euo pipefail`, which stops on a failed command, an unset variable, or a failed command within a pipeline. The command itself is produced from the Python specification rather than copied by hand, so the displayed design and executable script cannot silently diverge.

The cell below is opt-in because it creates files. When enabled, it writes:

- `run_generated_sphere.sh`;
- `run_generated_core_shell.sh`;
- `run_generated_linear_pearls.sh`.

It also adds executable permission. A script can then be started from a McStas-configured terminal with, for example, `./run_generated_sphere.sh`. Existing files are not overwritten unless `OVERWRITE_SCRIPTS` is also enabled. Before a production run, first reduce `histories` and `levels` in `sweep_specs` for a smoke test and inspect its NeXus output.

In [ ]:
WRITE_SCRIPTS = True
OVERWRITE_SCRIPTS = False

if WRITE_SCRIPTS:
    for filename, content in generated_scripts.items():
        destination = ROOT / filename
        if destination.exists() and not OVERWRITE_SCRIPTS:
            print(f'Skipped existing {destination.name}')
            continue
        destination.write_text(content)
        destination.chmod(destination.stat().st_mode | 0o111)
        print(f'Wrote executable {destination.name}')
else:
    print('Dry run only. Set WRITE_SCRIPTS=True to create the three shell scripts.')

### Run a small smoke test directly from Python

The same command builder can launch a test without a shell script. Here a copied sphere specification is reduced to two values per parameter and only 10,000 neutron histories. Execution remains disabled by default. Avoid testing the full production specification first: compilation or parameter errors should be discovered cheaply.

In [ ]:
smoke_spec = {**sweep_specs['sphere'], 'levels': 2}
smoke_command = make_multirun_command(smoke_spec, histories=10_000, mpi=2)
print(shlex.join(smoke_command))
print('Expected simulations:', smoke_spec['levels'] ** len(smoke_spec['ranges']))

RUN_SMOKE_TEST = True
if RUN_SMOKE_TEST:
    if shutil.which('mcrun') is None:
        raise RuntimeError('mcrun is not available in this environment')
    subprocess.run(smoke_command, check=True, cwd=ROOT)
else:
    print('Dry run only. Set RUN_SMOKE_TEST=True to launch it.')

### Scaling to a production system

For large datasets, generate a design table with one row per simulation and give every row a stable `simulation_id`. Run rows as scheduler array jobs, write each job to a temporary location, validate it, then move it into the final dataset. Store status and failure reason so interrupted runs can resume. Independent files are often easier to parallelize; a later consolidation step can create shards of manageable size. Avoid thousands of workers writing to one HDF5 file concurrently.

Record the instrument source and hash, McStas/McXtrace version, component versions, command, random seed, neutron histories, host/scheduler information, units, and creation time.

## Discover the NeXus/HDF5 schema

NeXus is an HDF5 convention, so `h5py` is enough for robust inspection. The local sample-model files store entries such as `/entry1/data/PSD_scattering_psd/data`; the AnySample files use `/entry1/data/PSD_dat/data`. A reusable loader should discover or configure the signal path rather than assume a single detector name.

In [ ]:
def entry_names(h5):
    names = [name for name in h5 if name.startswith('entry')]
    return sorted(names, key=lambda name: int(name.removeprefix('entry')))

def scalar_value(dataset):
    value = np.asarray(dataset[()]).reshape(-1)[0]
    if isinstance(value, (bytes, np.bytes_)):
        value = value.decode()
    try:
        return float(value)
    except (TypeError, ValueError):
        return value

h5_files = sorted(ROOT.glob('*/mccode.h5'))
print(f'Found {len(h5_files)} NeXus files')
for path in h5_files:
    with h5py.File(path, 'r') as h5:
        names = entry_names(h5)
        print(f'{path.parent.name:45s} {len(names):4d} entries')

In [ ]:
def describe_entry(path, entry_name='entry1'):
    with h5py.File(path, 'r') as h5:
        entry = h5[entry_name]
        params = {k: scalar_value(v) for k, v in entry['simulation/Param'].items()}
        signals = {
            name: group['data'].shape
            for name, group in entry['data'].items()
            if isinstance(group, h5py.Group) and 'data' in group
        }
    return params, signals

example_path = next(
    (p for p in h5_files if 'kws_sphere_' in p.parent.name),
    h5_files[0],
)
parameters, signals = describe_entry(example_path)
print('File:', example_path)
print('Parameters:', parameters)
print('Signals:', signals)

In [ ]:
def find_2d_signal(entry):
    candidates = []
    for name, group in entry['data'].items():
        if isinstance(group, h5py.Group) and 'data' in group and group['data'].ndim == 2:
            candidates.append(f'data/{name}/data')
    if len(candidates) != 1:
        raise ValueError(f'Expected one 2-D detector signal, found {candidates}')
    return candidates[0]

with h5py.File(example_path, 'r') as h5:
    first = h5[entry_names(h5)[0]]
    signal_path = find_2d_signal(first)
    image = np.asarray(first[signal_path], dtype=np.float32)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image, origin='lower')
axes[0].set_title('Raw counts/intensity')
axes[1].imshow(np.log1p(np.clip(image, 0, None)), origin='lower')
axes[1].set_title('log(1 + clipped signal)')
for ax in axes: ax.set(xlabel='detector x pixel', ylabel='detector y pixel')
plt.tight_layout()

## Validate before training

A successful simulation process is not automatically a valid ML example. 

Check: expected entry count; identical shapes and units; finite and nonnegative values where appropriate; signal that is not entirely zero; parameter bounds and combinations; duplicated rows; missing or failed jobs; class balance; and plots from random points and parameter-space boundaries. Also verify that train/validation/test splitting cannot put correlated replicates into different sets.

In [ ]:
def validate_nexus(path):
    issues, shapes = [], set()
    with h5py.File(path, 'r') as h5:
        names = entry_names(h5)
        if not names:
            return {'file': str(path), 'entries': 0, 'shapes': [], 'issues': ['no entries']}
        for name in names:
            entry = h5[name]
            try:
                signal_path = find_2d_signal(entry)
                image = np.asarray(entry[signal_path])
                shapes.add(image.shape)
                if not np.isfinite(image).all(): issues.append(f'{name}: non-finite values')
                if np.nanmax(image) <= 0: issues.append(f'{name}: no positive signal')
            except Exception as exc:
                issues.append(f'{name}: {exc}')
        if len(shapes) > 1:
            issues.append(f'inconsistent detector shapes: {sorted(shapes)}')
    return {'file': str(path), 'entries': len(names), 'shapes': sorted(shapes), 'issues': issues}

reports = [validate_nexus(path) for path in h5_files]
for report in reports:
    status = 'OK' if not report['issues'] else f'{len(report["issues"])} issue(s)'
    print(f'{Path(report["file"]).parent.name:45s} {report["entries"]:4d} {status}')

In [ ]:
# Rough storage planning for an uncompressed float32 detector tensor
detector_shape = image.shape
examples = 1_000_000
raw_gib = examples * np.prod(detector_shape) * np.dtype('float32').itemsize / 1024**3
print(f'{examples:,} images of shape {detector_shape}: about {raw_gib:,.1f} GiB before metadata/compression')
print('Benchmark real compression and I/O: sparse/noisy detector data can behave very differently.')

## Connect parameter variation to later training

Choosing parameters is part of defining the learning problem, not only part of running the simulator. The following discussion separates the information contributed by sample and instrument variation, explains the danger of correlating them, and connects these decisions to model inputs, targets, robustness, and evaluation. A matched-configuration implementation follows the discussion.

### Sample variation and instrument variation carry different information

A detector image is produced jointly by the sample and the instrument. Sample parameters describe the material or structure we usually want to infer, while instrument parameters describe how that structure is illuminated and observed. A useful AI dataset must represent both sources of variability deliberately.

### Information introduced by sample parameters

The sample family—sphere, core–shell, or linear pearls in this example—changes the underlying scattering model and is therefore the classification label. Continuous sample parameters change the signal within a class:

- `radius` changes characteristic feature positions and length scales;
- `pd_radius` controls polydispersity and can smooth or broaden features;
- `thickness` changes the relationship between core and shell scattering;
- `edge_sep` changes correlations between pearls and therefore the structure factor.

Varying these parameters teaches a classifier which features remain characteristic of a class despite within-class diversity. It also determines whether continuous parameters can later be regression targets. If the sample space is too narrow, the model may memorize a few ideal patterns. If it is unrealistically broad, different classes may become physically ambiguous. Bounds and distributions therefore need scientific justification.

### Information introduced by instrument parameters

Instrument parameters do not normally change the sample itself. They change the measurement transfer function, resolution, accessible range, intensity, noise, and detector representation. Examples include wavelength, wavelength spread, collimation, slit setting, sample-to-detector distance, pixel size, beam center, flux, background, and neutron count.

Instrument variation teaches the model that the same sample can produce different measured images under different configurations. This is essential when deployment data will come from several configurations or beamlines. Without it, a model may perform well only at the single simulated setting. However, adding instrument variability can make the learning problem harder because sample information may be blurred, shifted, cropped, or hidden by noise.

Some instrument values may be supplied to the network as metadata when they are known during inference. Others may be treated as nuisance variables: they are varied during training but hidden from the network so it must learn invariance. This decision must match the intended deployment workflow.

### Cross sample and instrument variation to avoid confounding

The most dangerous dataset design is one where sample class and instrument configuration are correlated. For example, if every sphere is simulated at one wavelength and every core–shell sample at another, a classifier can achieve high accuracy by identifying wavelength-dependent detector features without learning the sample physics. This shortcut fails as soon as configurations change.

For classification, each sample family should therefore experience the same distribution of relevant instrument configurations. A fully crossed design evaluates every sample design under every instrument configuration; this gives excellent separation of effects but can be expensive. A matched design is cheaper: corresponding rows from every class receive the same configuration distribution. Random independent assignment is acceptable only when the dataset is large enough and balance is verified explicitly.

### Consequences for splitting and evaluation

A random row split answers only whether the model can interpolate among configurations already represented in training. Stronger evaluations deliberately hold out sample ranges, instrument settings, or both. Useful complementary tests include:

- unseen sample parameters within the trained physical range;
- sample parameters outside the trained range;
- an unseen instrument configuration;
- familiar samples measured with unfamiliar combinations of settings;
- higher noise, background, masking, or calibration uncertainty;
- measured experimental data that were never used for model selection.

Store sample parameters, instrument parameters, seeds, and replicate-group identifiers separately in the manifest. This makes it possible to build these splits later, audit accidental correlations, condition the model on selected metadata, and report performance as a function of both sample and instrument configuration rather than as one average accuracy.

The KWS sample instruments expose `SlitSetting` and `Lam`. The next cell assigns the same configuration distribution to every sample family: rows with the same numeric index receive identical wavelength and slit settings in all three classes. This prevents these settings from becoming an accidental classification shortcut.

In [ ]:
configuration_rng = np.random.default_rng(31415)
matched_configurations = [
    {
        'Lam': float(configuration_rng.uniform(4.0, 10.0)),
        'SlitSetting': int(configuration_rng.choice([1, 2, 3])),
    }
    for _ in range(samples_per_class)
]

design_with_instrument_variation = []
for row in design:
    row_index = int(row['simulation_id'].rsplit('_', 1)[1])
    design_with_instrument_variation.append({
        **row,
        **matched_configurations[row_index],
    })

for family in ('sphere', 'core_shell', 'linear_pearls'):
    family_rows = [r for r in design_with_instrument_variation if r['sample_family'] == family]
    slit_counts = {setting: sum(r['SlitSetting'] == setting for r in family_rows) for setting in (1, 2, 3)}
    print(f'{family:14s}: {len(family_rows)} rows, slit counts {slit_counts}')

design_with_instrument_variation[:3]

## Validate parameter bounds and detect duplicate simulations

Detector validation alone cannot detect a corrupted or accidentally repeated design. The enhanced validator below checks stored simulation parameters against expected bounds and constructs a parameter signature for every entry. Identical signatures indicate duplicate parameter combinations. Repeated combinations are acceptable when they intentionally use different random seeds to measure Monte Carlo variation; in that case include the seed in the signature or record a shared replicate group.

In [ ]:
PARAMETER_BOUNDS = {
    'radius': (10.0, 100.0),
    'pd_radius': (0.0, 0.1),
    'thickness': (10.0, 100.0),
    'edge_sep': (100.0, 400.0),
}

def validate_nexus_parameters(path, parameter_bounds):
    issues = []
    signatures = {}
    checked = 0
    with h5py.File(path, 'r') as h5:
        for entry_name in entry_names(h5):
            checked += 1
            params = h5[f'{entry_name}/simulation/Param']
            signature = []
            for parameter, (lower, upper) in parameter_bounds.items():
                if parameter not in params:
                    continue  # class-specific parameters need not exist in every file
                value = float(scalar_value(params[parameter]))
                signature.append((parameter, value))
                if not lower <= value <= upper:
                    issues.append(
                        f'{entry_name}: {parameter}={value:g} is outside [{lower:g}, {upper:g}]'
                    )
            signature = tuple(signature)
            if signature and signature in signatures:
                issues.append(
                    f'{entry_name}: duplicates parameters from {signatures[signature]}: {signature}'
                )
            elif signature:
                signatures[signature] = entry_name
    return {
        'file': str(path),
        'entries_checked': checked,
        'issues': issues,
    }

parameter_reports = [validate_nexus_parameters(path, PARAMETER_BOUNDS) for path in h5_files]
for report in parameter_reports:
    status = 'OK' if not report['issues'] else f'{len(report["issues"])} issue(s)'
    print(f'{Path(report["file"]).parent.name:45s} {status}')

# Inspect details rather than silently discarding duplicates.
example_issues = next((r['issues'] for r in parameter_reports if r['issues']), [])
example_issues[:5]

## Design interpolation and extrapolation evaluations

A random stratified split mainly measures interpolation: training and test examples occupy the same parameter ranges. An extrapolation split asks a harder and different question by reserving boundary regions that training never sees. Both are useful and should be reported separately.

The functions below operate on the balanced manifest. The interpolation alternative shuffles within each class and assigns the same proportions to every class. The extrapolation alternative reserves small and large radii for testing, uses only central radii for training and validation, and therefore measures generalization beyond the training support.

In [ ]:
def interpolation_split(rows, seed=123, train_fraction=0.70, validation_fraction=0.15):
    split_rng = np.random.default_rng(seed)
    result = {'train': [], 'validation': [], 'test_interpolation': []}
    for family in sorted({row['sample_family'] for row in rows}):
        family_rows = [row for row in rows if row['sample_family'] == family]
        order = split_rng.permutation(len(family_rows))
        train_end = int(train_fraction * len(order))
        validation_end = train_end + int(validation_fraction * len(order))
        result['train'].extend(family_rows[i] for i in order[:train_end])
        result['validation'].extend(family_rows[i] for i in order[train_end:validation_end])
        result['test_interpolation'].extend(family_rows[i] for i in order[validation_end:])
    return result

def extrapolation_split(rows, lower_test_edge=20.0, upper_test_edge=90.0, seed=123):
    split_rng = np.random.default_rng(seed)
    result = {'train': [], 'validation': [], 'test_extrapolation': []}
    for family in sorted({row['sample_family'] for row in rows}):
        family_rows = [row for row in rows if row['sample_family'] == family]
        boundary = [row for row in family_rows if row['radius'] < lower_test_edge or row['radius'] > upper_test_edge]
        interior = [row for row in family_rows if lower_test_edge <= row['radius'] <= upper_test_edge]
        split_rng.shuffle(interior)
        validation_count = max(1, int(0.15 * len(interior)))
        result['validation'].extend(interior[:validation_count])
        result['train'].extend(interior[validation_count:])
        result['test_extrapolation'].extend(boundary)
    return result

interpolation = interpolation_split(design_with_instrument_variation)
extrapolation = extrapolation_split(design_with_instrument_variation)
for name, rows in interpolation.items():
    counts = {family: sum(r['sample_family'] == family for r in rows) for family in sweep_specs}
    print(f'interpolation/{name:18s}: {len(rows):4d} {counts}')
for name, rows in extrapolation.items():
    counts = {family: sum(r['sample_family'] == family for r in rows) for family in sweep_specs}
    print(f'extrapolation/{name:18s}: {len(rows):4d} {counts}')

## Exercise for the day

Think about the scientific purpose first, then plan and generate a large simulation dataset for another McStas instrument of your choice—or for an instrument you construct yourself. Use the provided HPC resources when the required coverage or neutron count makes local generation impractical.

Your project should justify the AI task, inputs, targets, nuisance variables, parameter bounds, sampling design, class balance, fidelity, storage estimate, validation rules, leakage-safe splits, and success criteria. 

Begin with a small smoke test, benchmark representative simulations, estimate CPU-hours and storage, and only then submit the full campaign. 

Preserve the instrument source, manifest, seeds, environment information, SLURM scripts, logs, validation reports, and dataset version so another student/person can reproduce the result.